# Avance 2 – Modelo Relacional
**Proyecto:** TechCore – DataVision Analytics

Este notebook toma el dataset limpio exportado desde Power Query (`ventasTransformed.csv`) y construye el modelo relacional completo:

`Sucursales`, `Vendedores`, `Ciudades`, `Clientes`, `Productos`, `DetalleFacturas`, `Facturas`

Incluye: generación de claves primarias y foráneas, validaciones de integridad referencial, reportes exploratorios y exportación final a `modeloVentas.xlsx` para su carga en Power BI.


## 1. Carga del dataset

In [1]:
# Importación de librerías
import pandas as pd
import numpy as np

# Cargar el CSV limpio exportado desde Power Query
# encoding='utf-8-sig' porque el archivo trae BOM (Byte Order Mark) al inicio
df = pd.read_csv('../avance_1_limpieza/ventasTransformed.csv', encoding='utf-8-sig')

df.head()

,VentaID,FechaVenta,HoraVenta,AñoVenta,MesVenta,MesYAñoVenta,SucursalNombre,CiudadSucursal,VendedorNombre,ClienteNombre,...,CantidadProducto2,PrecioUnitarioProducto2,SubtotalProducto2,NombreProducto3,MarcaProducto3,CantidadProducto3,PrecioUnitarioProducto3,SubtotalProducto3,DescuentoVenta,TotalVenta
0,1,31/12/2015,05:42 a. m.,2015,Diciembre,12/2015,TechCore Pereira,Pereira,Amílcar Ortega-Alberto,Bienvenida Nebot-Fiol,...,2,8000000,16000000,Dell Latitude 7420,Dell,1,5600000,5600000,0.0,31200000
1,2,23/03/2019,07:03 p. m.,2019,Marzo,03/2019,TechCore Medellín #1,Medellín,Ana Sofía Llopis Blázquez,Teófila Bueno-Novoa,...,1,6800000,6800000,No aplica,No aplica,0,0,0,0.0,10800000
2,3,23/12/2018,02:32 a. m.,2018,Diciembre,12/2018,TechCore Medellín #2,Medellín,Juan José Porcel-Riera,Gilberto Chamorro Catalá,...,1,7200000,7200000,HP Pavilion 15,HP,1,3500000,3500000,0.0,21900000
3,4,14/11/2015,12:37 a. m.,2015,Noviembre,11/2015,TechCore Medellín #2,Medellín,Juan José Porcel-Riera,Máximo Coronado Huerta,...,2,4400000,8800000,No aplica,No aplica,0,0,0,0.0,13600000
4,5,29/11/2016,10:34 a. m.,2016,Noviembre,11/2016,TechCore Cali,Cali,Jacinta Juárez Marín,Yago Oliver,...,1,3500000,3500000,Microsoft Surface Laptop 4,Microsoft,1,5200000,5200000,0.0,13900000


In [2]:
# Revisión rápida de dimensiones y columnas disponibles
print("Filas:", df.shape[0])
print("Columnas:", df.shape[1])
df.columns.tolist()

Filas: 30000
Columnas: 33


['VentaID',
 'FechaVenta',
 'HoraVenta',
 'AñoVenta',
 'MesVenta',
 'MesYAñoVenta',
 'SucursalNombre',
 'CiudadSucursal',
 'VendedorNombre',
 'ClienteNombre',
 'GeneroCliente',
 'EdadCliente',
 'EmailCliente',
 'TelefonoCliente',
 'DireccionCliente',
 'MetodoPago',
 'NombreProducto1',
 'MarcaProducto1',
 'CantidadProducto1',
 'PrecioUnitarioProducto1',
 'SubtotalProducto1',
 'NombreProducto2',
 'MarcaProducto2',
 'CantidadProducto2',
 'PrecioUnitarioProducto2',
 'SubtotalProducto2',
 'NombreProducto3',
 'MarcaProducto3',
 'CantidadProducto3',
 'PrecioUnitarioProducto3',
 'SubtotalProducto3',
 'DescuentoVenta',
 'TotalVenta']

## 2. Normalización de tipos de datos

`FechaVenta` y `HoraVenta` llegan como texto desde el CSV, y algunas columnas de dinero se leyeron como enteros. Se corrigen antes de construir el modelo, para que las tablas finales ya salgan con los tipos correctos hacia Power BI.

In [3]:
# Normalizar FechaVenta: texto 'DD/MM/AAAA' -> datetime
df['FechaVenta'] = pd.to_datetime(df['FechaVenta'], format='%d/%m/%Y')

# Normalizar HoraVenta: texto "hh:mm a. m./p. m." -> datetime.time (24h)
hora_limpia = (df['HoraVenta']
               .str.replace('a. m.', 'AM', regex=False)
               .str.replace('p. m.', 'PM', regex=False))
df['HoraVenta'] = pd.to_datetime(hora_limpia, format='%I:%M %p').dt.time

In [4]:
# Verificación de tipos de datos
df.dtypes

VentaID                             int64
FechaVenta                 datetime64[ns]
HoraVenta                          object
AñoVenta                            int64
MesVenta                           object
MesYAñoVenta                       object
SucursalNombre                     object
CiudadSucursal                     object
VendedorNombre                     object
ClienteNombre                      object
GeneroCliente                      object
EdadCliente                         int64
EmailCliente                       object
TelefonoCliente                    object
DireccionCliente                   object
MetodoPago                         object
NombreProducto1                    object
MarcaProducto1                     object
CantidadProducto1                   int64
PrecioUnitarioProducto1             int64
SubtotalProducto1                   int64
NombreProducto2                    object
MarcaProducto2                     object
CantidadProducto2                 

In [5]:
# Las columnas de dinero se leyeron como int64 porque no había decimales
# en el CSV, pero conceptualmente son montos -> se convierten a float
columnas_dinero = [
    "PrecioUnitarioProducto1", "SubtotalProducto1",
    "PrecioUnitarioProducto2", "SubtotalProducto2",
    "PrecioUnitarioProducto3", "SubtotalProducto3",
    "TotalVenta"
]

df[columnas_dinero] = df[columnas_dinero].astype(float)

# Confirmación
df[columnas_dinero].dtypes

PrecioUnitarioProducto1    float64
SubtotalProducto1          float64
PrecioUnitarioProducto2    float64
SubtotalProducto2          float64
PrecioUnitarioProducto3    float64
SubtotalProducto3          float64
TotalVenta                 float64
dtype: object

## 3. Función genérica para generar tablas dimensión

El dataset original está "aplanado": el nombre de cada sucursal, vendedor, cliente, etc. se repite en cada fila de venta donde aparece. Para armar cada tabla dimensión (una fila única por entidad, con su propio ID) se sigue siempre el mismo patrón:

1. Seleccionar la(s) columna(s) de esa entidad y quitar duplicados (`drop_duplicates`).
2. Reiniciar el índice para que quede consecutivo, sin huecos (`reset_index`).
3. Generar un ID consecutivo a partir de ese índice (`index + 1`).
4. Eliminar la columna auxiliar `index` que deja `reset_index`.
5. Reordenar columnas dejando el ID primero.

En vez de repetir estos 5 pasos a mano por cada entidad, se define una función reutilizable.

In [6]:
def crear_id(df, columna_nombre, columna_id):
    """
    Genera una tabla dimensión (valores únicos + clave sustituta) a partir
    de una o varias columnas de `df`.

    Parámetros
    ----------
    df : DataFrame origen
    columna_nombre : list[str] -> columnas que identifican a la entidad.
                     Se pasa siempre como lista, incluso si es una sola columna,
                     para que la función sirva tanto para entidades simples
                     (Sucursal, Vendedor) como compuestas (Cliente).
    columna_id : str -> nombre que tendrá la clave sustituta generada
                 

    Retorna
    -------
    DataFrame con el ID como primera columna, seguido de las columnas
    originales, sin duplicados.
    """
    tabla_unica = df[columna_nombre].drop_duplicates()
    tabla_unica_reset = tabla_unica.reset_index()
    tabla_unica_reset[columna_id] = tabla_unica_reset.index + 1
    tabla_unica_reset = tabla_unica_reset.drop(columns='index')
    tabla_unica_reset = tabla_unica_reset[[columna_id] + columna_nombre]
    return tabla_unica_reset

## 4. Generación de tablas dimensión

Se aplica `crear_id()` a cada entidad. Aunque la entidad tenga una sola columna de nombre (ej. Sucursal), se le pasa igual como **lista de un elemento**, porque así lo espera la función.

In [7]:
vendedores = crear_id(df, ['VendedorNombre'], 'VendedorID')
ciudades   = crear_id(df, ['CiudadSucursal'], 'CiudadID')

# Clientes lleva varias columnas de atributos, no solo el nombre
clientes = crear_id(df, ['ClienteNombre', 'GeneroCliente', 'EdadCliente',
                          'TelefonoCliente', 'EmailCliente', 'DireccionCliente'],
                     'ClienteID')

In [8]:
vendedores.head()

,VendedorID,VendedorNombre
0,1,Amílcar Ortega-Alberto
1,2,Ana Sofía Llopis Blázquez
2,3,Juan José Porcel-Riera
3,4,Jacinta Juárez Marín
4,5,Clímaco Salinas Zabaleta


In [9]:
ciudades.head()

,CiudadID,CiudadSucursal
0,1,Pereira
1,2,Medellín
2,3,Cali
3,4,Bogotá


In [10]:
clientes.head()

,ClienteID,ClienteNombre,GeneroCliente,EdadCliente,TelefonoCliente,EmailCliente,DireccionCliente
0,1,Bienvenida Nebot-Fiol,F,37,+34806548767,bienvenida19@hotmail.com,cll 52 #32-98
1,2,Teófila Bueno-Novoa,F,43,+34 947255990,teófila35@gmail.com,cll 96 #81-94
2,3,Gilberto Chamorro Catalá,M,38,+34 978 810 249,gilberto57@hotmail.com,cra 28 #79-85
3,4,Máximo Coronado Huerta,M,30,+34 825429634,máximo65@hotmail.com,cll 5 #89-26
4,5,Yago Oliver,M,51,+34988285275,yago38@yahoo.com,cra 32 #97-86


In [11]:
# Verificación rápida: el ID debe ser único en cada tabla dimensión
print("Vendedores:", vendedores['VendedorID'].is_unique, "-", len(vendedores), "filas")
print("Ciudades:  ", ciudades['CiudadID'].is_unique, "-", len(ciudades), "filas")
print("Clientes:  ", clientes['ClienteID'].is_unique, "-", len(clientes), "filas")

Vendedores: True - 30 filas
Ciudades:   True - 4 filas
Clientes:   True - 17453 filas


## 5. De formato ancho a formato largo: productos por línea

El dataset original tiene hasta 3 productos "estirados" en columnas por cada venta (`NombreProducto1..3`, `MarcaProducto1..3`, `CantidadProducto1..3`, `PrecioUnitarioProducto1..3`, `SubtotalProducto1..3`). Para el modelo relacional, cada línea de producto debe ser **una fila independiente**.

Se recorre cada posición (1, 2, 3) con un loop, seleccionando y renombrando las columnas correspondientes, y se apilan las 3 tablas resultantes con `pd.concat`.

In [12]:
lista = []

for i in range(1, 4):
    # Columnas de esta posición de producto (más VentaID como ancla)
    columnas_a_seleccionar = [
        'VentaID',
        f'NombreProducto{i}', f'MarcaProducto{i}', f'CantidadProducto{i}',
        f'PrecioUnitarioProducto{i}', f'SubtotalProducto{i}'
    ]
    tabla_temporal = df[columnas_a_seleccionar]

    # Se renombran las columnas quitando el número, para que las 3 tablas
    # (posición 1, 2 y 3) queden con exactamente los mismos nombres
    # y se puedan apilar sin problema
    tabla_temporal = tabla_temporal.rename(columns={
        f"NombreProducto{i}": "Nombre",
        f"MarcaProducto{i}": "Marca",
        f"CantidadProducto{i}": "Cantidad",
        f"PrecioUnitarioProducto{i}": "Precio",
        f"SubtotalProducto{i}": "Subtotal"
    })

    lista.append(tabla_temporal)

# Debe haber 3 tablas en la lista (una por posición de producto)
len(lista)

3

In [13]:
# Apilar las 3 tablas (posición 1, 2 y 3) una debajo de otra
detalle_productos = pd.concat(lista)

print(detalle_productos.shape)
detalle_productos.head()

(90000, 6)


,VentaID,Nombre,Marca,Cantidad,Precio,Subtotal
0,1,Apple MacBook Pro 16,Apple,1,9600000.0,9600000.0
1,2,Acer Nitro 5,Acer,1,4000000.0,4000000.0
2,3,Dell Latitude 7420,Dell,2,5600000.0,11200000.0
3,4,Asus TUF Gaming A15,Asus,1,4800000.0,4800000.0
4,5,HP Spectre x360,HP,1,5200000.0,5200000.0


In [14]:
# No todas las ventas tienen 3 productos: las posiciones vacías quedaron
# marcadas como "No aplica" (con Cantidad y Subtotal en 0) -> se descartan,
# porque no representan una línea de venta real
detalle_productos = detalle_productos[detalle_productos['Nombre'] != 'No aplica']

detalle_productos.shape

(60059, 6)

In [15]:
# El índice quedó con huecos después de filtrar -> se reinicia y se
# genera DetalleID como clave primaria de esta tabla (una fila = una línea
# de producto vendido dentro de una factura)
detalle_productos = detalle_productos.reset_index()
detalle_productos['DetalleID'] = detalle_productos.index + 1
detalle_productos = detalle_productos.drop(columns='index')
detalle_productos = detalle_productos[['DetalleID', 'VentaID', 'Nombre', 'Marca',
                                        'Cantidad', 'Precio', 'Subtotal']]

detalle_productos.head()

,DetalleID,VentaID,Nombre,Marca,Cantidad,Precio,Subtotal
0,1,1,Apple MacBook Pro 16,Apple,1,9600000.0,9600000.0
1,2,2,Acer Nitro 5,Acer,1,4000000.0,4000000.0
2,3,3,Dell Latitude 7420,Dell,2,5600000.0,11200000.0
3,4,4,Asus TUF Gaming A15,Asus,1,4800000.0,4800000.0
4,5,5,HP Spectre x360,HP,1,5200000.0,5200000.0


In [16]:
# Verificación: DetalleID debe ser único
print("DetalleID único:", detalle_productos['DetalleID'].is_unique)
print("Filas totales:", len(detalle_productos))

DetalleID único: True
Filas totales: 60059


## 6. Control de calidad: inconsistencia en `Marca`

Al generar la tabla `Productos`, aparecieron productos duplicados con el **mismo nombre y precio**, pero con `Marca` distinta: `"Razer"` vs `"Razor"`. Se comprobó contra `df` original que el typo viene del archivo fuente (columna `MarcaProducto3`) y no fue corregido en la limpieza de Power Query.

A diferencia de un cambio real de precio en el tiempo (que sí podría justificar 2 registros), aquí ambas filas tenían el precio idéntico — es un error de tipeo, no una diferencia real de negocio. Se corrige antes de generar `ProductoID`, para no arrastrar 2 identificadores distintos para el mismo producto.

In [17]:
# Corrección del typo "Razor" -> "Razer" antes de generar ProductoID
detalle_productos['Marca'] = detalle_productos['Marca'].str.replace('Razor', 'Razer', regex=False)

## 7. Tabla `Productos`

Se genera con la misma función `crear_id()`, tomando como base `detalle_productos` (ya con el typo corregido), usando `Nombre + Marca + Precio` como criterio de unicidad.

In [18]:
productos = crear_id(detalle_productos, ['Nombre', 'Marca', 'Precio'], 'ProductoID')

print(productos.shape)
print("Nombres duplicados:", productos['Nombre'].duplicated().sum())
productos.head()

(40, 4)
Nombres duplicados: 0


,ProductoID,Nombre,Marca,Precio
0,1,Apple MacBook Pro 16,Apple,9600000.0
1,2,Acer Nitro 5,Acer,4000000.0
2,3,Dell Latitude 7420,Dell,5600000.0
3,4,Asus TUF Gaming A15,Asus,4800000.0
4,5,HP Spectre x360,HP,5200000.0


## 8. Tabla `Sucursales` (con su relación a `Ciudades`)

La consigna no lo pide explícitamente, pero para que el modelo sea consistente, `Sucursales` necesita una clave foránea `CiudadID` que apunte a `Ciudades` (cada sucursal pertenece a una ciudad). Sin esto, `Ciudades` quedaría desconectada del resto del modelo.

Pasos:
1. Extraer de `df` la relación única `SucursalNombre` ↔ `CiudadSucursal`.
2. Cruzar esa relación con `ciudades` para traer `CiudadID`.
3. Generar `sucursales` con `crear_id()` y cruzarla con el resultado anterior para agregarle `CiudadID`.

In [19]:
# Paso 1: relación única sucursal-ciudad, tal como aparece en el dataset
relacion_sucursal_ciudad = df[['SucursalNombre', 'CiudadSucursal']].drop_duplicates()

# Paso 2: traer CiudadID desde la tabla ciudades
relacion_sucursal_ciudad = relacion_sucursal_ciudad.merge(ciudades, on='CiudadSucursal', how='left')

relacion_sucursal_ciudad

,SucursalNombre,CiudadSucursal,CiudadID
0,TechCore Pereira,Pereira,1
1,TechCore Medellín #1,Medellín,2
2,TechCore Medellín #2,Medellín,2
3,TechCore Cali,Cali,3
4,TechCore Bogotá #2,Bogotá,4
5,TechCore Bogotá #1,Bogotá,4


In [20]:
# Paso 3: generar Sucursales y agregarle CiudadID
sucursales = crear_id(df, ['SucursalNombre'], 'SucursalID')
sucursales = sucursales.merge(relacion_sucursal_ciudad, on='SucursalNombre', how='left')
sucursales = sucursales[['SucursalID', 'SucursalNombre', 'CiudadID']]

print(sucursales.shape)
sucursales

(6, 3)


,SucursalID,SucursalNombre,CiudadID
0,1,TechCore Pereira,1
1,2,TechCore Medellín #1,2
2,3,TechCore Medellín #2,2
3,4,TechCore Cali,3
4,5,TechCore Bogotá #2,4
5,6,TechCore Bogotá #1,4


## 9. `DetalleFacturas`: cruzar `ProductoID` y traer el `Descuento`

`detalle_productos` todavía tiene `Nombre`, `Marca`, `Precio`, `Subtotal` repetidos en cada línea — información redundante ahora que existe la tabla `Productos`. Se reemplaza por `ProductoID` mediante `merge`, usando las mismas 3 columnas con las que se generó ese ID (así el cruce es único).

También se agrega `Descuento`: en el dataset original (`DescuentoVenta`) es una **tasa por factura completa** (0%, 5%, 10% o 15%; un solo valor por `VentaID`, no por producto). Se decidió, como diseño, incluirlo igualmente en `DetalleFacturas` tal como lo pide la consigna, aunque conceptualmente sea un dato a nivel de factura (queda repetido en cada línea de una misma venta).

In [21]:
# Cruce para traer ProductoID (how='left' para detectar, no perder,
# cualquier línea que no encontrara producto)
detalle_facturas = detalle_productos.merge(
    productos, on=['Nombre', 'Marca', 'Precio'], how='left'
)

print(detalle_facturas.shape)
print("ProductoID sin encontrar:", detalle_facturas['ProductoID'].isna().sum())
detalle_facturas.head()

(60059, 8)
ProductoID sin encontrar: 0


,DetalleID,VentaID,Nombre,Marca,Cantidad,Precio,Subtotal,ProductoID
0,1,1,Apple MacBook Pro 16,Apple,1,9600000.0,9600000.0,1
1,2,2,Acer Nitro 5,Acer,1,4000000.0,4000000.0,2
2,3,3,Dell Latitude 7420,Dell,2,5600000.0,11200000.0,3
3,4,4,Asus TUF Gaming A15,Asus,1,4800000.0,4800000.0,4
4,5,5,HP Spectre x360,HP,1,5200000.0,5200000.0,5


In [22]:
# Traer el descuento por venta desde df (solo las 2 columnas necesarias,
# no todo df, para no arrastrar columnas de sobra)
descuentos = df[['VentaID', 'DescuentoVenta']]

detalle_facturas = detalle_facturas.merge(descuentos, on='VentaID', how='left')

print(detalle_facturas.shape)
print("Descuentos sin encontrar:", detalle_facturas['DescuentoVenta'].isna().sum())

(60059, 9)
Descuentos sin encontrar: 0


In [23]:
# Renombrar y dejar solo las columnas que pide el modelo
detalle_facturas = detalle_facturas.rename(columns={
    'VentaID': 'FacturaID',
    'DescuentoVenta': 'Descuento'
})
detalle_facturas = detalle_facturas[['DetalleID', 'FacturaID', 'ProductoID', 'Cantidad', 'Descuento']]

detalle_facturas.head()

,DetalleID,FacturaID,ProductoID,Cantidad,Descuento
0,1,1,1,1,0.0
1,2,2,2,1,0.0
2,3,3,3,2,0.0
3,4,4,4,1,0.0
4,5,5,5,1,0.0


## 10. Tabla `Facturas`

Se parte de `df` (una fila por venta) y se cruza contra las 3 tablas dimensión (`sucursales`, `clientes`, `vendedores`) para reemplazar cada nombre por su ID.

**Caso especial — clientes con nombre repetido:** existen clientes distintos que coinciden en `ClienteNombre` pero difieren en edad, teléfono, email o dirección (personas distintas con el mismo nombre, no un error de captura). Cruzar solo por `ClienteNombre` genera ambigüedad y duplica filas de `Facturas`. La solución es cruzar usando las **mismas columnas** con las que se generó `ClienteID` (`ClienteNombre + GeneroCliente + EdadCliente + TelefonoCliente + EmailCliente + DireccionCliente`), igual que se hizo en el propio `crear_id()`. Antes de aplicar esto se verificó que `VendedorNombre` sí es único en `Vendedores`, así que ese cruce puede hacerse solo por nombre sin riesgo.

In [24]:
# Selección base desde df: una fila por venta, con todos los atributos
# de cliente necesarios para un cruce único (no ambiguo)
facturas = df[['VentaID', 'FechaVenta', 'SucursalNombre',
                'ClienteNombre', 'GeneroCliente', 'EdadCliente',
                'TelefonoCliente', 'EmailCliente', 'DireccionCliente',
                'VendedorNombre']]

# Merge 1: Sucursal (solo el nombre es único en la tabla sucursales)
facturas = facturas.merge(sucursales[['SucursalID', 'SucursalNombre']],
                           on='SucursalNombre', how='left')

In [25]:
# Merge 2: Cliente. Se cruza por las 6 columnas completas, no solo el
# nombre, porque existen clientes distintos con el mismo nombre
facturas = facturas.merge(
    clientes,
    on=['ClienteNombre', 'GeneroCliente', 'EdadCliente',
        'TelefonoCliente', 'EmailCliente', 'DireccionCliente'],
    how='left'
)

print(facturas.shape)  # debe seguir en 30000 filas, sin duplicarse

(30000, 12)


In [26]:
# Verificación previa: confirmar que VendedorNombre sí es único en
# Vendedores antes de cruzar solo por nombre
print("Vendedores con nombre repetido:", vendedores['VendedorNombre'].duplicated().sum())

Vendedores con nombre repetido: 0


In [27]:
# Merge 3: Vendedor
facturas = facturas.merge(vendedores, on='VendedorNombre', how='left')

print(facturas.shape)
facturas.columns.tolist()

(30000, 13)


['VentaID',
 'FechaVenta',
 'SucursalNombre',
 'ClienteNombre',
 'GeneroCliente',
 'EdadCliente',
 'TelefonoCliente',
 'EmailCliente',
 'DireccionCliente',
 'VendedorNombre',
 'SucursalID',
 'ClienteID',
 'VendedorID']

In [28]:
# Renombrar y dejar solo las columnas del modelo
facturas = facturas.rename(columns={
    'VentaID': 'FacturaID',
    'FechaVenta': 'Fecha'
})
facturas = facturas[['FacturaID', 'Fecha', 'SucursalID', 'ClienteID', 'VendedorID']]

print(facturas.shape)
facturas.head()

(30000, 5)


,FacturaID,Fecha,SucursalID,ClienteID,VendedorID
0,1,2015-12-31,1,1,1
1,2,2019-03-23,2,2,2
2,3,2018-12-23,3,3,3
3,4,2015-11-14,3,4,3
4,5,2016-11-29,4,5,4


In [29]:
# Verificación: no debe haber ningún ID sin encontrar
facturas.isna().sum()

FacturaID     0
Fecha         0
SucursalID    0
ClienteID     0
VendedorID    0
dtype: int64

## 11. Validación de integridad referencial

Se revisan las 6 relaciones del modelo: por cada una, se buscan filas cuya clave foránea **no** exista en la tabla referenciada (`~ ... .isin(...)`).

```
DetalleFacturas.FacturaID   -> Facturas.FacturaID
DetalleFacturas.ProductoID  -> Productos.ProductoID
Facturas.SucursalID         -> Sucursales.SucursalID
Facturas.ClienteID          -> Clientes.ClienteID
Facturas.VendedorID         -> Vendedores.VendedorID
Sucursales.CiudadID         -> Ciudades.CiudadID
```

In [30]:
huerfanos_factura  = detalle_facturas[~detalle_facturas['FacturaID'].isin(facturas['FacturaID'])]
huerfanos_producto = detalle_facturas[~detalle_facturas['ProductoID'].isin(productos['ProductoID'])]
huerfanos_sucursal = facturas[~facturas['SucursalID'].isin(sucursales['SucursalID'])]
huerfanos_cliente  = facturas[~facturas['ClienteID'].isin(clientes['ClienteID'])]
huerfanos_vendedor = facturas[~facturas['VendedorID'].isin(vendedores['VendedorID'])]
huerfanos_ciudad   = sucursales[~sucursales['CiudadID'].isin(ciudades['CiudadID'])]

print("Huérfanos en FacturaID:", len(huerfanos_factura))
print("Huérfanos en ProductoID:", len(huerfanos_producto))
print("Huérfanos en SucursalID:", len(huerfanos_sucursal))
print("Huérfanos en ClienteID:", len(huerfanos_cliente))
print("Huérfanos en VendedorID:", len(huerfanos_vendedor))
print("Huérfanos en CiudadID:  ", len(huerfanos_ciudad))

Huérfanos en FacturaID: 0
Huérfanos en ProductoID: 0
Huérfanos en SucursalID: 0
Huérfanos en ClienteID: 0
Huérfanos en VendedorID: 0
Huérfanos en CiudadID:   0


## 12. Reportes exploratorios

Se usa `detalle_facturas` cruzada con `productos` (para tener `Marca` y `Precio` disponibles) como base de ambos reportes.

In [31]:
# Base para los reportes: detalle + atributos de producto
total_ventas_marca = detalle_facturas.merge(productos, on='ProductoID', how='left')

# Formato de números: comas de miles, sin notación científica
pd.options.display.float_format = '{:,.0f}'.format

print(total_ventas_marca.shape)
total_ventas_marca.head()

(60059, 8)


,DetalleID,FacturaID,ProductoID,Cantidad,Descuento,Nombre,Marca,Precio
0,1,1,1,1,0,Apple MacBook Pro 16,Apple,"9,600,000"
1,2,2,2,1,0,Acer Nitro 5,Acer,"4,000,000"
2,3,3,3,2,0,Dell Latitude 7420,Dell,"5,600,000"
3,4,4,4,1,0,Asus TUF Gaming A15,Asus,"4,800,000"
4,5,5,5,1,0,HP Spectre x360,HP,"5,200,000"


### 12.1 Total de ventas (en $) por marca

In [32]:
# Subtotal por línea = Cantidad x Precio (no viene ya calculado en esta tabla)
total_ventas_marca['Subtotal'] = total_ventas_marca['Cantidad'] * total_ventas_marca['Precio']

ventas_por_marca = total_ventas_marca.groupby('Marca')['Subtotal'].sum().sort_values(ascending=False)
ventas_por_marca

Marca
Lenovo      117,879,200,000
HP           96,786,700,000
Dell         86,722,000,000
Apple        78,016,200,000
Asus         33,179,600,000
Acer         22,000,400,000
Samsung      12,760,800,000
MSI           8,525,200,000
Microsoft     8,223,400,000
Razer         3,936,400,000
Name: Subtotal, dtype: float64

### 12.2 Top 10 productos más vendidos (por unidades)

In [33]:
top_productos = total_ventas_marca.groupby('Nombre')['Cantidad'].sum().sort_values(ascending=False)
top_productos.head(10)

Nombre
HP Spectre x360              7726
Lenovo Legion 5 Pro          5861
Lenovo ThinkPad X1 Carbon    5814
Lenovo Yoga 7i               5052
HP Omen 16                   5011
Lenovo IdeaPad 5             4970
Dell XPS 13                  4526
Dell Latitude 7420           4507
HP Pavilion 15               4029
Dell Alienware m15           3658
Name: Cantidad, dtype: int64

## 13. Exportación a `modeloVentas.xlsx`

Cada tabla del modelo se exporta como una hoja separada, lista para cargarse en Power BI. `index=False` en todas para no arrastrar el índice interno de pandas como si fuera una columna de datos.

In [ ]:
with pd.ExcelWriter("modeloVentas.xlsx", engine="xlsxwriter") as writer:
    facturas.to_excel(writer, sheet_name="Facturas", index=False)
    detalle_facturas.to_excel(writer, sheet_name="DetalleFacturas", index=False)
    productos.to_excel(writer, sheet_name="Productos", index=False)
    clientes.to_excel(writer, sheet_name="Clientes", index=False)
    sucursales.to_excel(writer, sheet_name="Sucursales", index=False)
    ciudades.to_excel(writer, sheet_name="Ciudades", index=False)
    vendedores.to_excel(writer, sheet_name="Vendedores", index=False)

print("Archivo modeloVentas.xlsx generado con éxito.")